# Fine-tune MiniLM tren AllNLI pair-score 500k


## 1. Cai thu vien

In [1]:
%pip install -q -U "datasets>=3.0" "sentence-transformers>=5.0,<6" "accelerate>=1.0" "scikit-learn>=1.3" "pandas>=2.0" "pyarrow>=15.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.

## 2. Import va cau hinh

In [2]:
import json
import os
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.losses import CosineSimilarityLoss
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, average_precision_score, precision_recall_curve, precision_recall_fscore_support, roc_auc_score

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TRAIN_DATASET_ID = "sentence-transformers/all-nli"
BENCHMARK_DATASET_ID = "phdquang/allnli-pair-class-processed"

OUTPUT_ROOT = Path("/kaggle/working/allnli-pair-score-500k-minilm")
FINAL_MODEL_DIR = OUTPUT_ROOT / "final"
RESULTS_DIR = OUTPUT_ROOT / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_TRAIN_SAMPLES = 500_000
EVAL_PAIR_SCORE_SAMPLES = 10_000
NUM_EPOCHS = 1
TRAIN_BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRADIENT_ACCUMULATION_STEPS = 1
# LR thap hon 2e-5 de tranh lam hong pretrained space.
LEARNING_RATE = 5e-6
WARMUP_RATIO = 0.1
MAX_SEQ_LENGTH = 128
EVAL_STEPS = 1000
SAVE_STEPS = 1000
MAX_RETRIEVAL_QUERIES = 1000
RETRIEVAL_POOL_SIZE = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU chua duoc bat. Kaggle Settings > Accelerator > GPU, roi restart session.")

print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUTPUT_ROOT)

GPU: Tesla T4
Output: /kaggle/working/allnli-pair-score-500k-minilm


## 3. Load AllNLI pair-score 500k de train

In [3]:
train_score = load_dataset(TRAIN_DATASET_ID, "pair-score", split="train")
print(train_score)
print(train_score.column_names)
print(train_score[0])

train_score = train_score.shuffle(seed=SEED)
if MAX_TRAIN_SAMPLES and len(train_score) > MAX_TRAIN_SAMPLES:
    train_score = train_score.select(range(MAX_TRAIN_SAMPLES))

keep_columns = ["sentence1", "sentence2", "score"]
remove_columns = [col for col in train_score.column_names if col not in keep_columns]
if remove_columns:
    train_score = train_score.remove_columns(remove_columns)

train_score = train_score.filter(
    lambda row: bool(str(row["sentence1"]).strip()) and bool(str(row["sentence2"]).strip())
)
print("Train pair-score rows:", len(train_score))
print(pd.Series(train_score["score"]).value_counts().sort_index())

README.md: 0.00B [00:00, ?B/s]

pair-score/train-00000-of-00001.parquet:   0%|          | 0.00/69.5M [00:00<?, ?B/s]

pair-score/dev-00000-of-00001.parquet:   0%|          | 0.00/1.57M [00:00<?, ?B/s]

pair-score/test-00000-of-00001.parquet:   0%|          | 0.00/1.61M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/942069 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/19657 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/19656 [00:00<?, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'score'],
    num_rows: 942069
})
['sentence1', 'sentence2', 'score']
{'sentence1': 'A person on a horse jumps over a broken down airplane.', 'sentence2': 'A person is training his horse for a competition.', 'score': 0.5}


Filter:   0%|          | 0/500000 [00:00<?, ? examples/s]

Train pair-score rows: 500000
0.0    166883
0.5    166251
1.0    166866
Name: count, dtype: int64


## 4. Evaluator pair-score/dev

In [4]:
pair_score_dev = load_dataset(TRAIN_DATASET_ID, "pair-score", split="dev")
pair_score_dev = pair_score_dev.shuffle(seed=SEED)
if EVAL_PAIR_SCORE_SAMPLES and len(pair_score_dev) > EVAL_PAIR_SCORE_SAMPLES:
    pair_score_dev = pair_score_dev.select(range(EVAL_PAIR_SCORE_SAMPLES))

evaluator = EmbeddingSimilarityEvaluator(
    sentences1=[str(x) for x in pair_score_dev["sentence1"]],
    sentences2=[str(x) for x in pair_score_dev["sentence2"]],
    scores=[float(x) for x in pair_score_dev["score"]],
    batch_size=EVAL_BATCH_SIZE,
    main_similarity="cosine",
    name="allnli-pair-score-dev",
    show_progress_bar=True,
)
print("Eval rows:", len(pair_score_dev))

Eval rows: 10000


## 5. Load benchmark pair-class cua project

In [5]:
ds = load_dataset(BENCHMARK_DATASET_ID)

train_ds = ds["train"]
dev_ds = ds["dev"]
test_ds = ds["test"]

TEXT_A = "premise_clean" if "premise_clean" in train_ds.column_names else "premise"
TEXT_B = "hypothesis_clean" if "hypothesis_clean" in train_ds.column_names else "hypothesis"

train_df = train_ds.to_pandas()
dev_df = dev_ds.to_pandas()
test_df = test_ds.to_pandas()

if "label_name" not in dev_df.columns:
    label_map = {0: "entailment", 1: "neutral", 2: "contradiction"}
    for frame in [train_df, dev_df, test_df]:
        frame["label_name"] = frame["label"].map(label_map)

print(ds)
print("Benchmark columns:", train_ds.column_names)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/960k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/880k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/906k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clean', 'hypothesis_clean', 'label_name', 'premise_char_len', 'hypothesis_char_len', 'premise_token_len', 'hypothesis_token_len', 'lexical_overlap'],
        num_rows: 5000
    })
    dev: Dataset({
        features: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clean', 'hypothesis_clean', 'label_name', 'premise_char_len', 'hypothesis_char_len', 'premise_token_len', 'hypothesis_token_len', 'lexical_overlap'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'premise_clean', 'hypothesis_clean', 'label_name', 'premise_char_len', 'hypothesis_char_len', 'premise_token_len', 'hypothesis_token_len', 'lexical_overlap'],
        num_rows: 5000
    })
})
Benchmark columns: ['dataset_name', 'subset', 'split', 'premise', 'hypothesis', 'label', 'pr

## 6. Ham benchmark

In [6]:
POSITIVE_LABEL = "entailment"
NEGATIVE_RETRIEVAL_LABEL = "contradiction"

def choose_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    index = int(np.argmax(f1))
    return float(thresholds[index]), float(f1[index])

def classification_metrics(y_true, scores, threshold):
    predictions = scores >= threshold
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, predictions, average="binary", zero_division=0)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
    }

def transformer_pair_scores(model, frame):
    left = model.encode(frame[TEXT_A].fillna("").astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    right = model.encode(frame[TEXT_B].fillna("").astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    return np.sum(left * right, axis=1)

def transformer_retrieval_metrics(model, frame, seed):
    positives = frame[frame["label_name"] == POSITIVE_LABEL].reset_index(drop=True)
    negatives = frame[frame["label_name"] == NEGATIVE_RETRIEVAL_LABEL].reset_index(drop=True)
    rng = np.random.default_rng(seed)
    if len(positives) > MAX_RETRIEVAL_QUERIES:
        positives = positives.iloc[rng.choice(len(positives), MAX_RETRIEVAL_QUERIES, replace=False)].reset_index(drop=True)
    queries = model.encode(positives[TEXT_B].astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    relevant = model.encode(positives[TEXT_A].astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    distractors_all = model.encode(negatives[TEXT_A].astype(str).tolist(), batch_size=EVAL_BATCH_SIZE, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    ranks = []
    for index, query in enumerate(queries):
        distractor_ids = rng.choice(len(distractors_all), RETRIEVAL_POOL_SIZE - 1, replace=False)
        scores = np.concatenate(([float(relevant[index] @ query)], distractors_all[distractor_ids] @ query))
        permutation = rng.permutation(RETRIEVAL_POOL_SIZE)
        relevant_position = int(np.flatnonzero(permutation == 0)[0])
        ranking = np.argsort(-scores[permutation], kind="stable")
        ranks.append(int(np.flatnonzero(ranking == relevant_position)[0]) + 1)
    ranks = np.asarray(ranks)
    return {
        "queries": int(len(ranks)),
        "precision_at_1": float(np.mean(ranks <= 1)),
        "recall_at_5": float(np.mean(ranks <= 5)),
        "mrr": float(np.mean(1.0 / ranks)),
        "mean_rank": float(np.mean(ranks)),
    }

def evaluate_transformer(name, model):
    dev_scores = transformer_pair_scores(model, dev_df)
    test_scores = transformer_pair_scores(model, test_df)
    dev_targets = (dev_df["label_name"] == POSITIVE_LABEL).to_numpy()
    test_targets = (test_df["label_name"] == POSITIVE_LABEL).to_numpy()
    threshold, _ = choose_threshold(dev_targets, dev_scores)
    result = {
        "model": name,
        **classification_metrics(test_targets, test_scores, threshold),
        **transformer_retrieval_metrics(model, test_df, SEED + 1),
    }
    return result, test_scores

## 7. Baselines tren benchmark

In [7]:
tfidf = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), max_features=50000, min_df=2, sublinear_tf=True, norm="l2")
tfidf.fit(pd.concat([train_df[TEXT_A], train_df[TEXT_B]], ignore_index=True).fillna(""))

def tfidf_scores(frame):
    left = tfidf.transform(frame[TEXT_A].fillna(""))
    right = tfidf.transform(frame[TEXT_B].fillna(""))
    return np.asarray(left.multiply(right).sum(axis=1)).ravel()

dev_targets = (dev_df["label_name"] == POSITIVE_LABEL).to_numpy()
test_targets = (test_df["label_name"] == POSITIVE_LABEL).to_numpy()
tfidf_dev_scores = tfidf_scores(dev_df)
tfidf_test_scores = tfidf_scores(test_df)
tfidf_threshold, _ = choose_threshold(dev_targets, tfidf_dev_scores)
tfidf_result = {"model": "TF-IDF", **classification_metrics(test_targets, tfidf_test_scores, tfidf_threshold)}

model = SentenceTransformer(BASE_MODEL, device="cuda")
model.max_seq_length = MAX_SEQ_LENGTH
pretrained_result, pretrained_test_scores = evaluate_transformer("Pretrained MiniLM", model)

pd.DataFrame([tfidf_result, pretrained_result])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

,model,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,queries,precision_at_1,recall_at_5,mrr,mean_rank
0,TF-IDF,0.129885,0.5530,0.427704,0.800455,0.557513,0.666524,0.504354,NaN,NaN,NaN,NaN,NaN
1,Pretrained MiniLM,0.579734,0.6774,0.527568,0.794201,0.633991,0.780868,0.642787,1000.0,0.975,0.997,0.984458,1.054


## 8. Fine-tune bang CosineSimilarityLoss

In [8]:
bf16_supported = bool(hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported())

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_ROOT / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=not bf16_supported,
    bf16=bf16_supported,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    logging_strategy="steps",
    logging_steps=100,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

loss = CosineSimilarityLoss(model)
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_score,
    loss=loss,
    evaluator=evaluator,
)
trainer.train()

The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.
Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Allnli-pair-score-dev Pearson Cosine,Allnli-pair-score-dev Spearman Cosine
1000,0.101267,No log,0.634320,0.638834
2000,0.099392,No log,0.652319,0.655998
3000,0.095215,No log,0.659056,0.662672


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3907, training_loss=0.10015430523972697, metrics={'train_runtime': 1012.7055, 'train_samples_per_second': 493.727, 'train_steps_per_second': 3.858, 'total_flos': 0.0, 'train_loss': 0.10015430523972697, 'epoch': 1.0})

## 9. Luu model va benchmark

In [9]:
model.save_pretrained(str(FINAL_MODEL_DIR))
final_eval = evaluator(model, output_path=str(RESULTS_DIR))
finetuned_result, finetuned_test_scores = evaluate_transformer("Fine-tuned MiniLM pair-score-500k", model)

comparison = pd.DataFrame([tfidf_result, pretrained_result, finetuned_result])
comparison.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)

predictions = test_df[["premise", "hypothesis", "label_name"]].copy()
predictions["tfidf_cosine"] = tfidf_test_scores
predictions["pretrained_minilm_cosine"] = pretrained_test_scores
predictions["finetuned_minilm_pair_score_500k_cosine"] = finetuned_test_scores
predictions.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)

metadata = {
    "train_dataset": f"{TRAIN_DATASET_ID}/pair-score",
    "benchmark_dataset": BENCHMARK_DATASET_ID,
    "base_model": BASE_MODEL,
    "loss": "CosineSimilarityLoss",
    "requested_train_samples": MAX_TRAIN_SAMPLES,
    "actual_train_samples": len(train_score),
    "epochs": NUM_EPOCHS,
    "batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "warmup_ratio": WARMUP_RATIO,
    "max_seq_length": MAX_SEQ_LENGTH,
    "gpu": torch.cuda.get_device_name(0),
    "final_eval": {str(k): float(v) if hasattr(v, "item") else v for k, v in final_eval.items()},
}
(RESULTS_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

comparison

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

,model,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,queries,precision_at_1,recall_at_5,mrr,mean_rank
0,TF-IDF,0.129885,0.5530,0.427704,0.800455,0.557513,0.666524,0.504354,NaN,NaN,NaN,NaN,NaN
1,Pretrained MiniLM,0.579734,0.6774,0.527568,0.794201,0.633991,0.780868,0.642787,1000.0,0.975,0.997,0.984458,1.054
2,Fine-tuned MiniLM pair-score-500k,0.563581,0.7606,0.613673,0.862422,0.717088,0.866627,0.754212,1000.0,0.925,0.987,0.949809,1.257


In [10]:
model_zip = shutil.make_archive("/kaggle/working/allnli-pair-score-500k-minilm-model", "zip", FINAL_MODEL_DIR)
results_zip = shutil.make_archive("/kaggle/working/allnli-pair-score-500k-minilm-results", "zip", RESULTS_DIR)
print("Model ZIP:", model_zip)
print("Results ZIP:", results_zip)

Model ZIP: /kaggle/working/allnli-pair-score-500k-minilm-model.zip
Results ZIP: /kaggle/working/allnli-pair-score-500k-minilm-results.zip
